In [ ]:
!pip install unsloth

In [ ]:
# Importing The Base Model

from unsloth import FastLanguageModel

model , tokenizer = FastLanguageModel.from_pretrained(
    model_name = "Qwen/Qwen2.5-1.5B-Instruct",
    max_seq_length = 1024,
    load_in_4bit = True

)

In [ ]:
# Inferencing the Base Model

def chat(prompt):

  messages = [
      {
          "role" : "system", "content" : "You are a Helpful Assistant who helps users with their questions"
      },
      {
          "role" : "user" , "content" : str(prompt)
      }
  ]

  inputs = tokenizer.apply_chat_template(messages,tokenize = True, add_generation_prompt= True,   return_tensors = "pt", return_dict = True)
  inputs.to(model.device)
  output = model.generate(**inputs, max_new_tokens = 256)
  input_length = inputs['input_ids'].shape[1]
  new_tokens = output[0][input_length : ]
  decoded_output = tokenizer.decode(new_tokens, skip_special_tokens = True)
  print(decoded_output)

In [ ]:
prompt = "What is Claude Mythos"
chat(prompt)

In [ ]:
!unzip rag_db.zip

In [ ]:
!pip install chromadb

In [ ]:
# Inferencing the Base Model With RAG

import chromadb
client = chromadb.PersistentClient(path = "rag_db")
collection = client.get_collection(name = "AI-Information")

def chat_with_rag_connected(prompt):

  result = collection.query(
      query_texts = [prompt],
      n_results = 5
  )



  messages = [
      {
          "role" : "system", "content" : """Answer the user's question using the provided context.
                                            Do not invent facts or make unsupported claims.
                                            If the context does not contain enough information, say so.
                                            Answer clearly and concisely."""
      },
      {
          "role" : "user" , "content" : f"{str(prompt)}, Context 1 {result['documents'][0][0]}, Context 2 : {result['documents'][0][1]}"
      }
  ]

  inputs = tokenizer.apply_chat_template(messages,tokenize = True, add_generation_prompt= True,   return_tensors = "pt", return_dict = True)
  inputs.to(model.device)
  output = model.generate(**inputs, max_new_tokens = 256)
  input_length = inputs['input_ids'].shape[1]
  new_tokens = output[0][input_length : ]
  decoded_output = tokenizer.decode(new_tokens, skip_special_tokens = True)
  print(decoded_output)

In [ ]:
prompt = "What is Claude Mythos"

chat_with_rag_connected(prompt)